## 论文附录：统一实验代码（去重整合版）

本文件把原分散在 `notebooks/1`–`4` 中的**同一条研究流水线**合并为单一叙事：路径与随机种子只出现一次，`Dataset` / `BaselineLSTM` / `BiLSTMAttention` / 训练循环各保留**一份**实现。

**有意省略（避免附录堆砌）**：第 3 章大量统计制图、`2` 中 FinBERT 制图、`4` 的完整随机搜索双层循环、`5–8` 的复现搜索/消融/第 5 章画图/差分目标补充等，仍以原编号 Notebook 为准；此处用**最小自检**验证模型张量形状。

运行目录：在 `notebooks/` 下打开本文件执行（与项目其余 Notebook 一致）。


In [ ]:
from __future__ import annotations

import ast
import copy
import gc
import os
import random
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm
from transformers import AutoModelForSequenceClassification, AutoTokenizer

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

ROOT = Path.cwd()
PROJECT_DIR = ROOT.parent if ROOT.name == "notebooks" else ROOT
DATA_DIR = PROJECT_DIR / "data"
MODEL_DIR = PROJECT_DIR / "models"
RESULT_DIR = PROJECT_DIR / "results"
for d in (DATA_DIR, MODEL_DIR, RESULT_DIR):
    d.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)


### 1. 原始数据对齐与目标变量（`01_preprocessed_data.csv`）


In [ ]:
def load_and_merge_data(base_path: Path, llm_path: Path) -> pd.DataFrame:
    df_base = pd.read_csv(base_path)
    df_llm = pd.read_csv(llm_path)
    df_base["timestamp"] = pd.to_datetime(df_base["timestamp"])
    df_llm["timestamp"] = pd.to_datetime(df_llm["timestamp"])
    df_base = df_base.sort_values("timestamp").reset_index(drop=True)
    df_llm = df_llm.sort_values("timestamp").reset_index(drop=True)
    llm_cols = ["timestamp", "reasoning_text", "sentiment_class", "action_class", "action_score"]
    return pd.merge(df_base, df_llm[llm_cols], on="timestamp", how="inner")


def preprocess_and_build_target(df: pd.DataFrame) -> pd.DataFrame:
    # 防穿越：仅用前向填充
    df = df.ffill()
    df = df.dropna(subset=["high", "low"]).reset_index(drop=True)
    H_t, L_t = df["high"], df["low"]
    eps = 1e-8
    V_P = (np.log(H_t / L_t)) ** 2 / (4 * np.log(2))
    sigma_t = np.where(V_P > 0, np.log(V_P + eps), np.nan)
    # 次日对数 Parkinson 波动率作为监督目标
    df["target_sigma_t_plus_1"] = pd.Series(sigma_t).shift(-1)
    return df.dropna(subset=["target_sigma_t_plus_1"]).reset_index(drop=True)


BASE_CSV = DATA_DIR / "merged_daily.csv"
LLM_CSV = DATA_DIR / "merged_daily_gemini-1.5-flash_opinion.csv"
OUT_01 = DATA_DIR / "01_preprocessed_data.csv"

merged = load_and_merge_data(BASE_CSV, LLM_CSV)
clean = preprocess_and_build_target(merged)
clean.to_csv(OUT_01, index=False)
print("saved", OUT_01, clean.shape)


### 2. FinBERT 日度情感得分（`02_finbert_features_added.csv`）


In [ ]:
def quick_extract(cell_value):
    if pd.isna(cell_value) or cell_value == "":
        return []
    try:
        parsed = ast.literal_eval(cell_value)
        seq = parsed if isinstance(parsed, list) else [parsed]
        out = []
        for item in seq:
            if isinstance(item, list) and len(item) > 4:
                out.append(str(item[4]))
            elif isinstance(item, (str, list)):
                out.append(str(item))
        return out
    except Exception:
        return []


def run_finbert_daily_scores(df: pd.DataFrame) -> pd.DataFrame:
    sentiment_map = {"positive": 1, "neutral": 0, "negative": -1}
    action_map = {"buy": 1, "hold": 0, "sell": -1}
    df = df.copy()
    df["sentiment_class"] = df["sentiment_class"].str.lower().map(sentiment_map).fillna(0)
    df["action_class"] = df["action_class"].str.lower().map(action_map).fillna(0)
    df["action_score"] = pd.to_numeric(df["action_score"], errors="coerce").ffill().fillna(7.5)

    text_cols = ["cointelegraph", "bitcoin_news", "reddit"]
    for col in text_cols:
        df[col] = df[col].apply(quick_extract)

    model_name = "ProsusAI/finbert"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name).to(device)
    model.eval()

    scores, batch_size = [], 16
    with torch.no_grad():
        for _, row in tqdm(df.iterrows(), total=len(df), desc="FinBERT"):
            daily_texts = row["cointelegraph"] + row["bitcoin_news"] + row["reddit"]
            if not daily_texts:
                scores.append(0.0)
                continue
            chunk_means = []
            for i in range(0, len(daily_texts), batch_size):
                batch = daily_texts[i : i + batch_size]
                inputs = tokenizer(
                    batch,
                    padding=True,
                    truncation=True,
                    max_length=512,
                    return_tensors="pt",
                ).to(device)
                logits = model(**inputs).logits
                probs = F.softmax(logits, dim=-1)
                chunk_means.append((probs[:, 0] - probs[:, 1]).mean().item())
            scores.append(float(np.mean(chunk_means)))
            if device.type == "cuda" and len(scores) % 200 == 0:
                torch.cuda.empty_cache()
                gc.collect()

    df["finbert_sentiment_score"] = scores
    return df


df_fb = pd.read_csv(DATA_DIR / "01_preprocessed_data.csv")
df_fb = run_finbert_daily_scores(df_fb)
df_fb.to_csv(DATA_DIR / "02_finbert_features_added.csv", index=False)
print("saved", DATA_DIR / "02_finbert_features_added.csv", df_fb.shape)


### 3. 建模特征矩阵 v2（`04_final_feature_matrix_v2.csv`）

与 `notebooks/3` 中矩阵构建逻辑一致；**描述统计、ADF、Granger、第 3 章插图**在原 Notebook 中执行，此处不重复。


In [ ]:
input_path = DATA_DIR / "02_finbert_features_added.csv"
df = pd.read_csv(input_path)

df["llm_composite_score"] = df["sentiment_class"] * (df["action_score"] / 10.0)
df["sigma_t"] = df["target_sigma_t_plus_1"].shift(1)
df = df.dropna(subset=["sigma_t"]).reset_index(drop=True)

G1_cols = [
    "open",
    "close",
    "high",
    "low",
    "volume",
    "blocks-size",
    "avg-block-size",
    "n-transactions-total",
    "n-transactions-per-block",
    "hash-rate",
    "difficulty",
    "miners-revenue",
    "transaction-fees-usd",
    "n-unique-addresses",
    "n-transactions",
    "estimated-transaction-volume-usd",
    "total-bitcoins",
    "market-cap",
    "fng_value",
    "cbbi_value",
    "sigma_t",
]
G2_cols = G1_cols + ["finbert_sentiment_score"]
G3_cols = G1_cols + ["llm_composite_score"]
G3_M_cols = G1_cols + ["sentiment_class", "action_class", "action_score"]

all_feature_cols = list(set(G1_cols + G2_cols + G3_cols + G3_M_cols))
meta_cols = ["timestamp", "target_sigma_t_plus_1"]
final_columns = meta_cols + all_feature_cols
df = df[final_columns].copy()

n = len(df)
train_end, val_end = int(n * 0.7), int(n * 0.8)
train_df, val_df, test_df = df.iloc[:train_end], df.iloc[train_end:val_end], df.iloc[val_end:]

scaler = StandardScaler()
scaler.fit(train_df[all_feature_cols])
scaled = df.copy()
scaled[all_feature_cols] = scaler.transform(df[all_feature_cols])

joblib.dump(scaler, MODEL_DIR / "feature_scaler_v2.pkl")
ordered = meta_cols + [c for c in final_columns if c not in meta_cols]
out_csv = DATA_DIR / "04_final_feature_matrix_v2.csv"
scaled[ordered].to_csv(out_csv, index=False)
print("saved", out_csv, scaled.shape)


### 4. 序列数据集、基线 LSTM、主模型 Bi-LSTM + 注意力、训练循环

训练逻辑与 `notebooks/4.search_train_v2.ipynb` 中 `_train_and_validate` 对齐：AdamW + ReduceLROnPlateau + 验证集早停。**完整超参随机搜索**仍放在 `4` 中执行，避免附录重复几十页 trial 循环。


In [ ]:
DATA_PATH = DATA_DIR / "04_final_feature_matrix_v2.csv"


class CryptoVolatilityDataset(Dataset):
    def __init__(self, feature_matrix: np.ndarray, target_array: np.ndarray, lookback_window: int = 7):
        self.features = feature_matrix
        self.targets = target_array
        self.window = lookback_window

    def __len__(self) -> int:
        return len(self.features) - self.window + 1

    def __getitem__(self, idx: int):
        x = self.features[idx : idx + self.window]
        y = self.targets[idx + self.window - 1]
        return torch.FloatTensor(x), torch.FloatTensor([y])


class BaselineLSTM(nn.Module):
    def __init__(self, input_size: int, hidden_size: int, dropout_p: float):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers=1, batch_first=True)
        self.dropout = nn.Dropout(dropout_p)
        self.fc = nn.Sequential(nn.Linear(hidden_size, 32), nn.ReLU(), nn.Linear(32, 1))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        out, _ = self.lstm(x)
        return self.fc(self.dropout(out[:, -1, :]))


class AttentionLayer(nn.Module):
    def __init__(self, hidden_dim: int):
        super().__init__()
        self.attention_weights = nn.Linear(hidden_dim, 1, bias=False)

    def forward(self, lstm_output: torch.Tensor):
        scores = self.attention_weights(lstm_output)
        alpha = F.softmax(scores, dim=1)
        context = torch.sum(alpha * lstm_output, dim=1)
        return context, alpha.squeeze(-1)


class BiLSTMAttention(nn.Module):
    def __init__(self, input_size: int, hidden_size: int, dropout_p: float):
        super().__init__()
        self.bilstm = nn.LSTM(
            input_size, hidden_size, num_layers=1, batch_first=True, bidirectional=True
        )
        lstm_out_dim = hidden_size * 2
        self.attention = AttentionLayer(hidden_dim=lstm_out_dim)
        self.dropout = nn.Dropout(p=dropout_p)
        self.fc_layers = nn.Sequential(nn.Linear(lstm_out_dim, 32), nn.ReLU(), nn.Linear(32, 1))

    def forward(self, x: torch.Tensor):
        lstm_out, _ = self.bilstm(x)
        context, attn = self.attention(lstm_out)
        return self.fc_layers(self.dropout(context)), attn


def feature_columns_for_group(df: pd.DataFrame, feature_group: str) -> list[str]:
    all_cols = list(df.columns)
    base = [
        c
        for c in all_cols
        if c
        not in {
            "timestamp",
            "target_sigma_t_plus_1",
            "finbert_sentiment_score",
            "llm_composite_score",
            "sentiment_class",
            "action_class",
            "action_score",
        }
    ]
    fg_map = {
        "G1": base,
        "G2": base + ["finbert_sentiment_score"],
        "G3": base + ["llm_composite_score"],
        "G3-M": base + ["sentiment_class", "action_class", "action_score"],
    }
    return fg_map[feature_group]


def train_with_early_stopping(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    *,
    is_attention: bool,
    lr: float,
    weight_decay: float,
    max_epochs: int = 300,
    patience: int = 20,
) -> tuple[nn.Module, float]:
    criterion = nn.MSELoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.5, patience=7, min_lr=1e-6
    )
    best_val, best_wts, bad = float("inf"), None, 0
    for _ in range(max_epochs):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            pred = model(xb)[0] if is_attention else model(xb)
            loss = criterion(pred, yb)
            loss.backward()
            optimizer.step()
        model.eval()
        vlosses = []
        with torch.no_grad():
            for xv, yv in val_loader:
                xv, yv = xv.to(device), yv.to(device)
                pv = model(xv)[0] if is_attention else model(xv)
                vlosses.append(criterion(pv, yv).item())
        avg_v = float(np.mean(vlosses))
        scheduler.step(avg_v)
        if avg_v < best_val:
            best_val, best_wts, bad = avg_v, copy.deepcopy(model.state_dict()), 0
        else:
            bad += 1
            if bad >= patience:
                break
    if best_wts is not None:
        model.load_state_dict(best_wts)
    return model, best_val


### 5. 结构自检（单 batch 前向，不跑随机搜索）


In [ ]:
dfm = pd.read_csv(DATA_PATH)
fg = "G3"
f_cols = feature_columns_for_group(dfm, fg)
te, ve = int(len(dfm) * 0.7), int(len(dfm) * 0.8)
train_ds = CryptoVolatilityDataset(
    dfm.iloc[:te][f_cols].values, dfm.iloc[:te]["target_sigma_t_plus_1"].values
)
loader = DataLoader(train_ds, batch_size=8, shuffle=True)
xb, yb = next(iter(loader))

m_main = BiLSTMAttention(len(f_cols), hidden_size=64, dropout_p=0.3).to(device)
m_main.eval()
with torch.no_grad():
    pred, w = m_main(xb.to(device))
print("Main  yhat:", pred.shape, "attn:", w.shape, "y:", yb.shape)

m_base = BaselineLSTM(len(f_cols), hidden_size=64, dropout_p=0.3).to(device)
m_base.eval()
with torch.no_grad():
    pb = m_base(xb.to(device))
print("Base  yhat:", pb.shape)
